# Autonomous UAV Policy Transfer for Robust Urban Infrastructure Inspection

This notebook is the paper-aligned version of `uav_inspection_drl.py`, converted into `.ipynb` format.

## What was updated with minimal disruption
1. Removed notebook-export artifacts and stray text that made the original exported `.py` invalid.
2. Kept the original ROS + Flightmare + ORB-SLAM3 environment wrapper, route generation utilities, and PPO OSD core.
3. Added fuzzy descriptors `F = [mu_T, mu_L, mu_W, mu_A]` to align the state and reward interfaces with the paper.
4. Added optional OP-CBRS reward-shaping hooks based on offline potential functions learned from uniform-speed tasks.
5. Added lightweight Sim2Sim fuzzy-rule recalibration utilities for source-to-target simulated domain transfer.
6. Removed hard-coded CUDA assumptions where possible to improve portability.

## Notes
- ROS-specific imports are still required at runtime for simulator interaction.
- Illumination and wind are implemented as configurable fuzzy proxies because the original message stream mainly exposes feature-map and IMU data.
- The code structure is intentionally kept close to the original workflow.


In [ ]:
# RL Env Encapsulation
import os
import sys
import time
import math
import numpy
import rospy
import signal
import rospkg
import roslaunch
import subprocess
import numpy as np
import geometry_msgs.msg as geometry_msgs
import quadrotor_msgs.msg as quadrotor_msgs
import std_msgs.msg as std_msgs
import random
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Sequence, Tuple
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from lib.Lie import *

class UavController:
    '''
    UavController class encapsulates UAV controller for flightmare environment.
    Handles ROS node initialization, publishers, subscribers, & UAV control commands.
    '''

    def __init__(self):
        # Initialize ROS node for UAV control
        rospy.init_node('uavController',anonymous=False)
        self._quad_namespace = None
        self._connected = False

        # Initialize publishers for various UAV commands
        self._arm_bridge_pub = None
        self._start_pub = None
        self._land_pub = None
        self._off_pub = None
        self._force_hover_pub = None
        self._go_to_pose_pub = None
        self._set_velo_pub = None

        # Initialize state message and timestamp for autopilot feedback
        self.state_msg = quadrotor_msgs.AutopilotFeedback()
        self._autopilot_feedback_stamp = rospy.Time.now()

        # Store the previous autopilot state
        self._previous_autopilot_state = self.state_msg.OFF

        # Current pose of the UAV
        self.curPose = [0.0,0.0,0.0,0.0]

        # Rate objects for controlling loop frequency
        self.rate50hz = rospy.Rate(50)
        self.rate1hz = rospy.Rate(1)

    def connect(self, quad_namespace):
        # Connect to the UAV namespace and set up publishers and subscribers
        self._quad_namespace = quad_namespace

        self._arm_bridge_pub = rospy.Publisher(
            quad_namespace+'/bridge/arm', std_msgs.Bool, queue_size=1)
        self._start_pub = rospy.Publisher(
            quad_namespace+'/autopilot/start', std_msgs.Empty, queue_size=1)
        self._land_pub = rospy.Publisher(
            quad_namespace+'/autopilot/land', std_msgs.Empty, queue_size=1)
        self._off_pub = rospy.Publisher(
            quad_namespace+'/autopilot/off', std_msgs.Empty, queue_size=1)
        self._force_hover_pub = rospy.Publisher(
            quad_namespace+'/autopilot/force_hover', std_msgs.Empty,
            queue_size=1)
        self._go_to_pose_pub = rospy.Publisher(
            quad_namespace+'/autopilot/pose_command', geometry_msgs.PoseStamped, queue_size=1)
        self._set_velo_pub = rospy.Publisher(
            quad_namespace+'/autopilot/velocity_command', geometry_msgs.TwistStamped, queue_size=50)

        # Subscribe to autopilot feedback
        self._autopilot_feedback_sub = rospy.Subscriber(
            quad_namespace+'/autopilot/feedback',
            quadrotor_msgs.AutopilotFeedback, self._autopilot_feedback_cb)
        self._connected = True

    def _disconnect_pub_sub(self, pub):
        # Disconnect a publisher or subscriber
        if pub is not None:
            pub.unregister()
            pub = None

    def _autopilot_feedback_cb(self, msg):
        # Callback function to update the current pose based on autopilot feedback
        x = msg.reference_state.pose.position.x
        y = msg.reference_state.pose.position.y
        z = msg.reference_state.pose.position.z
        w = msg.reference_state.heading
        self.curPose = [round(x,1),round(y,1),round(z,1),round(w,1)]
        self.state_msg = msg
        self._autopilot_feedback_stamp = rospy.Time.now()

    def disconnect(self):
        # Disconnect all publishers and subscribers
        self._disconnect_pub_sub(self._autopilot_feedback_sub)
        self._disconnect_pub_sub(self._arm_bridge_pub)
        self._disconnect_pub_sub(self._start_pub)
        self._disconnect_pub_sub(self._land_pub)
        self._disconnect_pub_sub(self._off_pub)
        self._disconnect_pub_sub(self._force_hover_pub)
        self._disconnect_pub_sub(self._go_to_pose_pub)
        self._connected = False

    def isConnnect(self):
        # Check if the UAV is connected
        return self._connected

    def arm_bridge(self):
        # Arm the UAV bridge
        arm_message = std_msgs.Bool(True)
        self._arm_bridge_pub.publish(arm_message)

    def start(self):
        # Start the UAV
        start_message = std_msgs.Empty()
        self._start_pub.publish(start_message)

    def land(self):
        # Land the UAV
        land_message = std_msgs.Empty()
        self._land_pub.publish(land_message)

    def powerOff(self):
        # Power off the UAV
        start_message = std_msgs.Empty()
        self._off_pub.publish(start_message)

    def forceHover(self):
        # Force the UAV to hover
        force_hover_msg = std_msgs.Empty()
        self._force_hover_pub.publish(force_hover_msg)

    def to_pose(self,x,y,z,w):
        # Move the UAV to a specific pose
        go_to_pose_msg = geometry_msgs.PoseStamped()
        go_to_pose_msg.pose.position.x = float(x)
        go_to_pose_msg.pose.position.y = float(y)
        go_to_pose_msg.pose.position.z = float(z)

        heading = float(w) / 180.0 * math.pi
        go_to_pose_msg.pose.orientation.w = math.cos(heading / 2.0)
        go_to_pose_msg.pose.orientation.z = math.sin(heading / 2.0)
        self._go_to_pose_pub.publish(go_to_pose_msg)

    def _pub_velo(self,x,y,z,ax,ay,az):
        # Publish velocity commands to the UAV
        set_velo_msg = geometry_msgs.TwistStamped()
        set_velo_msg.twist.linear.x = float(x)
        set_velo_msg.twist.linear.y = float(y)
        set_velo_msg.twist.linear.z = float(z)
        set_velo_msg.twist.angular.x = float(ax);
        set_velo_msg.twist.angular.y = float(ay);
        set_velo_msg.twist.angular.z = float(az);
        self._set_velo_pub.publish(set_velo_msg)

    def move_velo_time(self,x,y,z,ax,ay,az,t):
        # Move the UAV with a specific velocity for a given time duration
        for i in range(int(t*50)):
            self._pub_velo(x,y,z,ax,ay,az);
            self.rate50hz.sleep()

    def dash_to_pose(self,x,y,z):
        # Move the UAV to a specific pose with velocity control
        while(True):
            m = self.state_msg
            clip_lb = 1
            clip_ub = 10
            vx = x - m.reference_state.pose.position.x
            vx = np.clip(vx,clip_lb,clip_ub) if vx>0 else np.clip(vx,-clip_ub,-clip_lb)
            vy = y - m.reference_state.pose.position.y
            vy = np.clip(vy,clip_lb,clip_ub) if vy>0 else np.clip(vy,-clip_ub,-clip_lb)
            vz = z - m.reference_state.pose.position.z
            vz = np.clip(vz,clip_lb,clip_ub) if vz>0 else np.clip(vz,-clip_ub,-clip_lb)

            for i in range(3):
                self._pub_velo(vx,vy,vz,0,0,0);
                self.rate50hz.sleep()

            m = self.state_msg
            if (x - m.reference_state.pose.position.x)*(x - m.reference_state.pose.position.x)+\
                (y - m.reference_state.pose.position.y)*(y - m.reference_state.pose.position.y)+\
                (z - m.reference_state.pose.position.z)*(z - m.reference_state.pose.position.z)< 1:
                break

        while(self.get_autopilot_state_name() != "HOVER"):
            self.rate50hz.sleep()
        self.to_pose(x,y,z,0)

    def get_autopilot_state_name(self):
        # Get the current autopilot state as a string
        if (self.state_msg.autopilot_state == self.state_msg.START):
            return "START"
        if (self.state_msg.autopilot_state == self.state_msg.HOVER):
            return "HOVER"
        if (self.state_msg.autopilot_state == self.state_msg.LAND):
            return "LAND"
        if (self.state_msg.autopilot_state == self.state_msg.EMERGENCY_LAND):
            return "EMERGENCY_LAND"
        if (self.state_msg.autopilot_state == self.state_msg.BREAKING):
            return "BREAKING"
        if (self.state_msg.autopilot_state == self.state_msg.GO_TO_POSE):
            return "GO_TO_POSE"
        if (self.state_msg.autopilot_state == self.state_msg.VELOCITY_CONTROL):
            return "VELOCITY_CONTROL"
        if (self.state_msg.autopilot_state == self.state_msg.REFERENCE_CONTROL):
            return "REFERENCE_CONTROL"
        if (self.state_msg.autopilot_state == self.state_msg.TRAJECTORY_CONTROL):
            return "TRAJECTORY_CONTROL"
        if (self.state_msg.autopilot_state == self.state_msg.COMMAND_FEEDTHROUGH):
            return "COMMAND_FEEDTHROUGH"
        if (self.state_msg.autopilot_state == self.state_msg.RC_MANUAL):
            return "RC_MANUAL"
        return "OFF"

    def imuInitFlightPath(self):
        # Initialize the UAV's flight path using IMU data
        init_rate = rospy.Rate(1/2)
        v = 0.3
        self.move_velo_time(v,v,-v,0,0,0,5)
        init_rate.sleep()
        self.move_velo_time(-v,0,0,0,0,0,8)
        init_rate.sleep()
        self.move_velo_time(v,0,0,0,0,0,3)
        init_rate.sleep()
        v = 0.4
        self.move_velo_time(0,0,v,0,0,0,5)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()

class Env:
    '''
    Env class encapsulates the flightmare simulation environment.
    Handles the launching and shutting down of the simulation environment.
    '''

    def __init__(self):
        # Path to the flightmare environment launch file
        # !Modify the launchPath to your flightmare env launch file.

        self.launchPath = "/home/dbq/dbq/DRL_SLAM/env/flightmare_ws/src/flightmare/flightros/launch/pilot/rotors_gazebo.launch"

        self.uuid = roslaunch.rlutil.get_or_generate_uuid(None, False)
        # Initialize the ROS launch parent

        # roslaunch.parent.ROSLaunchParent.VERBOSE=False
        self.launchHandle = roslaunch.parent.ROSLaunchParent(self.uuid, [self.launchPath],verbose=False)
        self.uav = None
        self.task_trajectory = None
        self.uav_name = "/hummingbird"

        # Initial position of the UAV
        self.initPosition = [0.0,0.0,2.0,0.0]

        # SLAM progress
        self.slamPg = None

    def setup(self):
        # Set up the simulation environment
        roslaunch.configure_logging(self.uuid)
        self.launchHandle.start()

    def add_uavController(self):
        # Add a UAV controller to the environment
        self.uav = UavController()

        if not self.uav.isConnnect():
            self.uav.connect(self.uav_name)
            while not self.uav.isConnnect():
                pass

    def close(self):
        # Shutdown the simulation environment
        self.launchHandle.shutdown()
        self.uuid = roslaunch.rlutil.get_or_generate_uuid(None, False)
        self.launchHandle = roslaunch.parent.ROSLaunchParent(self.uuid, [self.launchPath])

    def reset(self):
        # Reset the UAV to its initial position
        if self.uav.get_autopilot_state_name() == 'OFF':
            rospy.loginfo("uav take-off ...")
            self.uav.rate1hz.sleep()
            self.uav.arm_bridge()
            self.uav.rate1hz.sleep()
            self.uav.start()
            while not self.isReset():
                self.uav.rate50hz.sleep()
            rospy.loginfo("The uav position is initialized successfully!")
        else:
            rospy.loginfo("Move the drone to the initialization position.")
            self.uav.dash_to_pose(0,0,2)
            while not self.isReset():
                self.uav.rate50hz.sleep()
            rospy.loginfo("The uav position is initialized successfully.")

    def reset_centry(self):
        # Reset the UAV to the center position
        self.uav.dash_to_pose(0,0,10)
        while self.uav.curPose != [0.0,0.0,10.0,0.0]:
            self.uav.rate50hz.sleep()

    def set_task_trajectory(self,goals):
        # Set the task trajectory for the UAV
        self.task_trajectory = goals

    def isReset(self):
        # Check if the UAV is reset to the initial position
        return self.uav.curPose == self.initPosition

    def slamLauncher(self):
        # Launch the SLAM process
        # Modify SLAM executable script path like: /home/dbq/dbq/DRL_SLAM/slam/exp/fm_orbslam3/stereoInertial
        self.slamPg = subprocess.Popen("sh run.sh",shell=True,cwd="/home/dbq/dbq/DRL_SLAM/slam/exp/fm_orbslam3/stereoInertial",start_new_session=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)

    def slamShutdown(self):
        # Shutdown the SLAM process
        os.killpg(os.getpgid(self.slamPg.pid),signal.SIGTERM)


## PPO / Test Environment / Fuzzy + OP-CBRS Alignment

This section contains the `TestEnv` wrapper, fuzzy state augmentation, OP-CBRS potential shaping hooks, and Sim2Sim transfer helpers added to align the implementation with the paper.


In [ ]:
# Env Test Code block
from concurrent.futures import ThreadPoolExecutor
import threading
from rlslam.msg import pixelstream
import geometry_msgs.msg as geometry_msgs
from geometry_msgs.msg import PointStamped, PoseStamped, TransformStamped

class avgPool10(nn.Module):
    def __init__(self):
        super(avgPool10,self).__init__()
        self.layer1 = nn.AvgPool2d(10,stride=10)
        self.layer2 = nn.MaxPool2d(3,stride=1,padding=1)

    def forward(self, x):
        x = torch.from_numpy(x)
        x = x.unsqueeze(0)
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.unsqueeze(0)
        return x

class avgPool30(nn.Module):
    def __init__(self):
        super(avgPool30,self).__init__()
        self.layer1 = nn.AvgPool2d(30,stride=30)

    def forward(self, x):
        x = torch.from_numpy(x)
        x = x.unsqueeze(0)
        x = self.layer1(x)
        x = torch.flatten(x)
        return x


@dataclass
class FuzzyDomainConfig:
    texture_low: float = 0.05
    texture_mid: float = 0.18
    texture_high: float = 0.35
    illumination_nominal: float = 0.80
    illumination_band: float = 0.20
    wind_nominal: float = 0.15
    wind_band: float = 0.20
    adherence_nominal: float = 0.80
    adherence_band: float = 0.20


class FuzzyInferenceEngine:
    """Lightweight fuzzy front-end used to align the original implementation
    with the paper's state augmentation and uncertainty-aware control.

    The original code exposes feature occupancy and IMU bias, but not direct
    illumination or wind sensors. Therefore, mu_L and mu_W are implemented as
    configurable proxies that can be overwritten by the caller during training
    or transfer experiments.
    """

    def __init__(self, cfg: Optional[FuzzyDomainConfig] = None):
        self.cfg = cfg or FuzzyDomainConfig()

    @staticmethod
    def _clip01(x: float) -> float:
        return float(np.clip(x, 0.0, 1.0))

    def texture_membership(self, feature_density: float) -> float:
        cfg = self.cfg
        if feature_density <= cfg.texture_low:
            return 0.0
        if feature_density >= cfg.texture_high:
            return 1.0
        return self._clip01((feature_density - cfg.texture_low) / max(cfg.texture_high - cfg.texture_low, 1e-8))

    def illumination_membership(self, illumination_proxy: float) -> float:
        cfg = self.cfg
        return self._clip01(1.0 - abs(illumination_proxy - cfg.illumination_nominal) / max(cfg.illumination_band, 1e-8))

    def wind_membership(self, wind_proxy: float) -> float:
        cfg = self.cfg
        return self._clip01(1.0 - abs(wind_proxy - cfg.wind_nominal) / max(cfg.wind_band, 1e-8))

    def adherence_membership(self, adherence_proxy: float) -> float:
        cfg = self.cfg
        return self._clip01(1.0 - abs(adherence_proxy - cfg.adherence_nominal) / max(cfg.adherence_band, 1e-8))

    def coverage_membership(self, mu_t: float, mu_l: float, mu_w: float) -> float:
        return self._clip01(0.45 * mu_t + 0.30 * mu_l + 0.25 * mu_w)

    def rescale_for_transfer(self, source_stats: Dict[str, float], target_stats: Dict[str, float]) -> None:
        s_tex = max(float(source_stats.get('texture_scale', 1.0)), 1e-6)
        t_tex = max(float(target_stats.get('texture_scale', 1.0)), 1e-6)
        ratio = t_tex / s_tex
        self.cfg.texture_low *= ratio
        self.cfg.texture_mid *= ratio
        self.cfg.texture_high *= ratio

        s_illum = max(float(source_stats.get('illumination_band', self.cfg.illumination_band)), 1e-6)
        t_illum = max(float(target_stats.get('illumination_band', self.cfg.illumination_band)), 1e-6)
        self.cfg.illumination_band *= t_illum / s_illum

        s_wind = max(float(source_stats.get('wind_band', self.cfg.wind_band)), 1e-6)
        t_wind = max(float(target_stats.get('wind_band', self.cfg.wind_band)), 1e-6)
        self.cfg.wind_band *= t_wind / s_wind


class PotentialFunctionLibrary:
    """Container for OP-CBRS potential functions learned from offline tasks."""

    def __init__(self, gamma: float = 0.95):
        self.gamma = gamma
        self.entries: List[Dict[str, object]] = []

    def add(self, name: str, phi_fn: Callable[[torch.Tensor, torch.Tensor], float], fixed_action: Optional[float] = None) -> None:
        self.entries.append({'name': name, 'phi_fn': phi_fn, 'fixed_action': fixed_action})

    def __len__(self) -> int:
        return len(self.entries)

    def select(self, action_value: float, memberships: Dict[str, float], recent_actions: Optional[Sequence[float]] = None) -> Optional[Dict[str, object]]:
        if not self.entries:
            return None
        if memberships.get('mu_T', 1.0) < 0.25:
            conservative = sorted([e for e in self.entries if e.get('fixed_action') is not None], key=lambda e: e['fixed_action'])
            if conservative:
                return conservative[0]
        candidates = []
        for entry in self.entries:
            fixed_action = entry.get('fixed_action')
            dist = 0.0 if fixed_action is None else abs(float(fixed_action) - float(action_value))
            candidates.append((dist, entry))
        candidates.sort(key=lambda x: x[0])
        return candidates[0][1]

    def shape_reward(self, base_reward: float, ft_state: torch.Tensor, imu_state: torch.Tensor, next_ft_state: torch.Tensor, next_imu_state: torch.Tensor, action_value: float, memberships: Dict[str, float], recent_actions: Optional[Sequence[float]] = None) -> Tuple[float, Dict[str, object]]:
        entry = self.select(action_value, memberships, recent_actions)
        if entry is None:
            return base_reward, {'phi_name': None, 'phi_s': 0.0, 'phi_next': 0.0, 'shaping': 0.0}
        phi_fn = entry['phi_fn']
        phi_s = float(phi_fn(ft_state, imu_state))
        phi_next = float(phi_fn(next_ft_state, next_imu_state))
        shaping = float(self.gamma * phi_s - phi_next)
        return float(base_reward + shaping), {'phi_name': entry['name'], 'phi_s': phi_s, 'phi_next': phi_next, 'shaping': shaping}

class TestEnv:
    def __init__(self,renv):
        self.env = renv
        self.avgPL = None
        self.agent = None
        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        self.fuzzy = FuzzyInferenceEngine()
        self.op_cbrs: Optional[PotentialFunctionLibrary] = None
        self.domain_context = {'illumination': 0.80, 'wind': 0.15}
        self.coverage_score = 0.0
        self.prev_coverage_score = 0.0
        self.coverage_reward_weight = 0.20
        self.recent_actions: List[float] = []
        # step函数连续控制独立线程
        self.stepThread = None
        self.threadPool = ThreadPoolExecutor(max_workers=2)
        self.future = self.threadPool.submit(self.stepVoid)

        # 在线校正
        self.recordAlignTraj = False
        self.alignTraj_ref = []
        self.alignTraj_est = []

        # 原始state 数据
        self.stateDict = {}
        self.last_ftState = None
        self.last_imuState = None

        # 维护 pose_est, pose_ref
        self.P_est = np.zeros(3)
        self.P_est_rot=np.zeros(4)
        self.P_est_ts = 0.0
        self.P_ref = np.zeros(3)#维护的是P_est对应的真值，每次更新P_est时都会更新P_ref
        self.P_ref_ts = 0.0
        self.P_ref_buf = {}
        self.P_gt = np.zeros(3)
        self.P_gt_rot=np.zeros(4)
        self.cbf = False # the clean buffer flag of P_ref_buf
        self.P_ref_buf_recordFlag = True
        self.curRpe = 0

        # 维护 一秒前的Pest Pref ,由于Pest的输出频率为20hz，因此维护一个长度为20的列表，并且设置一个index
        # 这个笔记本训练的policy 基于10hz的slam，步长为0.5秒。因此需要计算0.5s的rpe。window长度为5
        self.rpeWindow = [None for x in range(5)] # change : 10hz
        self.rpeIdx = 0
        # 维护一个逐帧rpe
        self.curFrame_Rpe = 0
        self.RpeFrameNum = 5
        self.cur_ape = 0
        # 维护世界坐标系原点与机坐标系原点
        self.TGU = None

        # 任务结束标志
        self.done = True

        # 奖励函数参数
        # Reward Function E -> [-1,0]
        self.curRe = 0
        self.rpe_mu=0.024
        self.rpe_sigma=0.0045
        self.RE_a = 1.0
        self.RE_b = self.rpe_mu
        self.RE_c = -1

        ## Reward Function G -> [-1,0]
        self.curRg = 0
        self.mid_action = 2.0    #动作中值
        self.action_lb = -0.5   #动作值下边界
        self.action_ub = 0.5    #动作值上边界
        self.action_ex_lb = self.mid_action + self.action_lb #执行动作值上边界
        self.action_ex_ub = self.mid_action + self.action_ub #执行动作值下边界
        ### 简单线性函数
        self.action = 0 # action = actor(state).tanh -> [-1,1]
        self.RG_a = 0.5
        self.RG_b = -0.5

        ## Reward weight
        self.WE = 2 # 2->3
        self.WG = 0.5 # 0.5
        self.deviatedReward = -1000


        ## error reward: 有时rpe会非常大：rpe>5 或者rpe为0 这些经验都不应该加入到buffer里
        self.rewardError = True
        self.rewardErrorCnt = 0
        self.reward = -1

        # slam launcher
        self.isSlamLaunchSucc = False

        # checkErrorBuffer
        self.checkErrorBuffer=[np.zeros(3),np.ones(3),np.zeros(3),np.ones(3),np.zeros(3)]
        self.checkId=0

        self.stateSub = None
        self.poseEstSub = None
        self.poseRefSub = None
        self.slamModePub = None
        self.ReAlignFlagPub = None

    def bind_agent(self, agent):
        self.agent = agent

    def set_domain_context(self, illumination: Optional[float] = None, wind: Optional[float] = None):
        if illumination is not None:
            self.domain_context['illumination'] = float(np.clip(illumination, 0.0, 1.0))
        if wind is not None:
            self.domain_context['wind'] = float(np.clip(wind, 0.0, 1.0))

    def attach_op_cbrs(self, potential_library: Optional[PotentialFunctionLibrary]):
        self.op_cbrs = potential_library

    def _feature_density_from_pixelstring(self, pixelstring: str) -> float:
        if not pixelstring:
            return 0.0
        items = [p for p in pixelstring.split(';') if p]
        return float(np.clip(len(items) / float(480 * 720), 0.0, 1.0))

    def _trajectory_adherence(self) -> float:
        err = float(np.linalg.norm(self.P_est - self.P_ref))
        return float(np.clip(1.0 - err / 2.0, 0.0, 1.0))

    def get_fuzzy_memberships(self) -> Dict[str, float]:
        pixelstring = self.stateDict.get('pixelstring', '') if self.stateDict else ''
        feature_density = self._feature_density_from_pixelstring(pixelstring)
        mu_t = self.fuzzy.texture_membership(feature_density)
        mu_l = self.fuzzy.illumination_membership(self.domain_context['illumination'])
        mu_w = self.fuzzy.wind_membership(self.domain_context['wind'])
        mu_a = self.fuzzy.adherence_membership(self._trajectory_adherence())
        mu_cvg = self.fuzzy.coverage_membership(mu_t, mu_l, mu_w)
        return {'feature_density': feature_density, 'mu_T': mu_t, 'mu_L': mu_l, 'mu_W': mu_w, 'mu_A': mu_a, 'mu_cvg': mu_cvg}

    def getRewardCoverageImprovement(self) -> float:
        memberships = self.get_fuzzy_memberships()
        self.prev_coverage_score = self.coverage_score
        self.coverage_score = memberships['mu_cvg']
        return self.coverage_reward_weight * (self.coverage_score - self.prev_coverage_score)

    def getRewardE(self):
        """计算误差奖励"""
        arpe = self.curRpe
        RE= self.RE_c*math.tanh(self.exp(self.func(arpe)))
        return RE

    def getRewardG(self):
        """用于计算进度奖励"""
        return self.RG_a*self.action+self.RG_b

    def stateCallback(self,msgs): # 训练一个网络的版本
        if self.stateDict != {} and msgs.imuBiasAx == 0: # imubias 为 0 则不更新
            self.stateDict['timestamp'] = msgs.timestamp
            self.stateDict['pixelNum'] = msgs.pixelNum
            self.stateDict['pixelstring'] = msgs.pixelstring
        else:
            self.stateDict = {
                'timestamp':msgs.timestamp,
                'pixelNum':msgs.pixelNum,
                'imuBiasAx':msgs.imuBiasAx,
                'imuBiasAy':msgs.imuBiasAy,
                'imuBiasAz':msgs.imuBiasAz,
                'imuBiasGx':msgs.imuBiasGx,
                'imuBiasGy':msgs.imuBiasGy,
                'imuBiasGz':msgs.imuBiasGz,
                'Vx':msgs.Vx,
                'Vy':msgs.Vy,
                'Vz':msgs.Vz,
                'pixelstring':msgs.pixelstring
            }

    def getCurState(self,dir):
        # dir = 0:up
        pixList = [p for p in self.stateDict['pixelstring'].split(';') if p]
        ftState = np.zeros((480,720),np.float32)
        if dir == 0:
            for onepixStr in pixList:
                pix = onepixStr.split(' ')
                ftState[int(pix[2])][int(pix[1])] = 1
        else:
            for onepixStr in pixList:
                pix = onepixStr.split(' ')
                ftState[479 - int(pix[2])][int(pix[1])] = 1
        last_ftState = self.avgPL(ftState).to(self.device)
        memberships = self.get_fuzzy_memberships()
        last_imuState = torch.from_numpy(np.array([
            float(self.action),
            float(self.stateDict['Vx']),
            float(self.stateDict['Vy']),
            float(self.stateDict['imuBiasAx']),
            float(self.stateDict['imuBiasAy']),
            float(self.stateDict['imuBiasAz']),
            float(memberships['mu_T']),
            float(memberships['mu_L']),
            float(memberships['mu_W']),
            float(memberships['mu_A'])]
        ).astype(np.float32)).unsqueeze(0).to(self.device)

        return last_ftState,last_imuState

    def getCurStateOffline(self):
        last_ftState = self.stateDict['pixelstring']
        memberships = self.get_fuzzy_memberships()
        last_imuState = torch.from_numpy(np.array([
            float(self.action),
            float(self.stateDict['Vx']),
            float(self.stateDict['Vy']),
            float(self.stateDict['imuBiasAx']),
            float(self.stateDict['imuBiasAy']),
            float(self.stateDict['imuBiasAz']),
            float(memberships['mu_T']),
            float(memberships['mu_L']),
            float(memberships['mu_W']),
            float(memberships['mu_A'])]
        ).astype(np.float32)).unsqueeze(0).to(self.device)

        return last_ftState,last_imuState


    def Pref_callback(self,msgs):
        """接收来自仿真环境的位姿真值msg。维护一个字典{时间戳，位姿}用于保存最近接收的真值，用于对齐位姿估计值与真值的时间戳"""
        self.P_gt=np.array([
                msgs.transform.translation.x,
                msgs.transform.translation.y,
                msgs.transform.translation.z])
        self.P_gt_rot=np.array([
                msgs.transform.rotation.x,
                msgs.transform.rotation.y,
                msgs.transform.rotation.z,
                msgs.transform.rotation.w
            ])

        if self.cbf:
            self.P_ref_buf = {}#用于存放最近50ms的位置的真值
            self.cbf = False
        if self.P_ref_buf_recordFlag:
            timestamp = msgs.header.stamp.secs+msgs.header.stamp.nsecs*1e-9
            self.P_ref_buf[timestamp] = np.array([msgs.transform.translation.x,msgs.transform.translation.y,msgs.transform.translation.z])
            self.P_ref_ts = timestamp

    def Pest_callback(self,msgs):
        """接收来自slam系统的位姿估计值msg，每接收一个位姿估计值，就和位姿真值进行匹配，并且计算RPE"""
        self.P_est_ts = msgs.header.stamp.secs+msgs.header.stamp.nsecs*1e-9
        self.P_est = np.array([msgs.pose.position.x,msgs.pose.position.y,msgs.pose.position.z])
        self.P_est_rot = np.array([
            msgs.pose.orientation.x,
            msgs.pose.orientation.y,
            msgs.pose.orientation.z,
            msgs.pose.orientation.w
            ])
        self.getP_ref(self.P_est_ts)
        # record Align Traj
        if self.recordAlignTraj:
            self.alignTraj_est.append(self.P_est)
            # print(self.P_est)
            # print(f'current alignSet lenght:{len(self.alignTraj_est)}')
            self.alignTraj_ref.append(self.P_gt)
        # calculate the Rpe
        self.curRpe = self.getRpe()


    def getP_ref(self,timestamp):
        """输入位姿估计值的时间戳，找到对应的位姿真值，并且更新P_ref，更新后的P_ref与P_est的时间戳是对齐的"""
        self.P_ref_buf_recordFlag = False

        pose_gt = self.P_ref_buf.get(timestamp)
        if pose_gt is None:
            # 当P_est 没有找到对应的 P_ret时，寻找最近的P_ref
            pose_gt = list(self.P_ref_buf.items())[-1][1]
        self.P_ref = pose_gt
        self.cbf = True
        self.P_ref_buf_recordFlag = True

    # tenv.setTGU()
    def setTGU(self,OriginPoint):
        tUO = OriginPoint[0:3]
        tGO = self.P_gt
        qUO = OriginPoint[3:7]
        qGO = self.P_gt_rot

        RMUO = quaternion2RotationMatrix(qUO)
        RMGO = quaternion2RotationMatrix(qGO)

        TMUO = np.eye(4)
        TMUO[0:3,0:3] = RMUO
        TMUO[0:3,3] = tUO
        TMGO = np.eye(4)
        TMGO[0:3,0:3] = RMGO
        TMGO[0:3,3] = tGO

        TUO = se3(matrix=TMUO)
        TGO = se3(matrix=TMGO)
        self.TGU = TGO + TUO.exp().inv()

    def calcTU_Ustar(self):
        tUP = self.P_est
        tGP = self.P_gt
        qUP = self.P_est_rot
        qGP = self.P_gt_rot

        RMUP = quaternion2RotationMatrix(qUP)
        RMGP = quaternion2RotationMatrix(qGP)

        TMUP = np.eye(4)
        TMUP[0:3,0:3] = RMUP
        TMUP[0:3,3] = tUP
        TMGP = np.eye(4)
        TMGP[0:3,0:3] = RMGP
        TMGP[0:3,3] = tGP

        TUP = se3(matrix=TMUP)
        TGP = se3(matrix=TMGP)
        TGU_star = TGP + TUP.exp().inv()
        TU_Ustar = self.TGU.exp().inv() + TGU_star
        return TU_Ustar

    def calcDeltaTold(self):
        PUP = np.hstack((self.P_est,self.getEstYaw()))
        TGP = se3(vector=np.array([
            0,
            0,
            math.pi/180 * self.getCurYaw(),
            self.P_gt[0],
            self.P_gt[1],
            self.P_gt[2]
        ]))
        TUP = se3(vector=np.array([
            0,
            0,
            math.pi/180 * PUP[3],
            PUP[0],
            PUP[1],
            PUP[2]
        ]))
        CutTGU = TGP + TUP.exp().inv()
        # DeltaT = np.linalg.norm(CutTGU.w - self.TGU.w)
        DeltaT = CutTGU + self.TGU.exp().inv()
        return DeltaT

    def getCurYaw(self):
        x,y,z,w=self.P_gt_rot
        siny_cosp = 2* (z*w+x*y)
        cosy_cosp = 1 - 2 * (y*y + z*z)
        yaw = math.atan2(siny_cosp,cosy_cosp)/math.pi*180
        return yaw

    def getEstYaw(self):
        x,y,z,w=self.P_est_rot
        siny_cosp = 2* (z*w+x*y)
        cosy_cosp = 1 - 2 * (y*y + z*z)
        yaw = math.atan2(siny_cosp,cosy_cosp)/math.pi*180
        return yaw

    def getRpe(self):# change : 10hz
        # print('query RPE')
        """计算当前位姿估计值的最近3帧平均rpe"""
        if self.rpeWindow[self.rpeIdx] is None:
            self.rpeWindow[self.rpeIdx] = [self.P_est,self.P_ref]
            self.rpeIdx = (self.rpeIdx+1)%5
            return 0
        else:
            d_est_4_0 = self.rpeWindow[(self.rpeIdx+4)%5][0] - self.rpeWindow[self.rpeIdx][0]
            d_est_5_0 = self.P_est - self.rpeWindow[self.rpeIdx][0]

            d_ref_4_0 = self.rpeWindow[(self.rpeIdx+4)%5][1] - self.rpeWindow[self.rpeIdx][1]
            d_ref_5_0 = self.P_ref - self.rpeWindow[self.rpeIdx][1]

            self.rpeWindow[self.rpeIdx] = [self.P_est,self.P_ref]
            self.rpeIdx = (self.rpeIdx+1)%5

            return (np.linalg.norm(d_est_4_0-d_ref_4_0)+\
                    np.linalg.norm(d_est_5_0-d_ref_5_0))/2

    def connect(self):
        """连接ros系统，即创建状态订阅器，位姿真值订阅器，位姿估计订阅器。创建一个slam定位策略发布者。"""
        # State Subscriber
        self.stateSub = rospy.Subscriber("/Stereo_Inertial/ORB_SLAM3_msg", pixelstream, self.stateCallback,queue_size=1,buff_size=32768)
        # Estimate Pose Subscriber
        self.poseRefSub = rospy.Subscriber("hummingbird/ground_truth/transform", geometry_msgs.TransformStamped, self.Pref_callback)
        # Ground Pose Subscriber
        self.poseEstSub = rospy.Subscriber("Stereo_Inertial/ORB_SLAM3_EstPose", geometry_msgs.PoseStamped, self.Pest_callback)
        # ReAlignFlag Publisher
        self.ReAlignFlagPub = rospy.Publisher('/ReAlignFlag', std_msgs.Int8, queue_size=1)

    def disconnect(self):
        """注销订阅者与发布者"""
        self.stateSub.unregister()
        self.poseRefSub.unregister()
        self.poseEstSub.unregister()
        self.ReAlignFlagPub.unregister()

    def stepIpml(self,NextPoint):
        xt,yt,zt,wt,orient=NextPoint
        x0,y0,z0 = self.P_est
        w0 = self.getEstYaw()
        vx = xt - x0
        vy = yt - y0
        vz = zt - z0
        vw = (wt - w0) * 1 / 60
        self.env.uav.move_velo_time(vx,vy,vz,0,0,vw,0.5)

    def stepVoid(self): #空动作
        pass

    def func(self,x):
        return self.RE_a*(x-self.RE_b)

    def exp(self,x):
        return math.exp(x)-1

    def step(self,NextPoint):
        """无人机智在 t_step 的时间内前往下一目标点
        :param NextPoint : (x,y,z,w,orient)
        :return: Next_State-下一个状态, Reward-该动作的奖励, isDone-是否完成任务
        """
        ret_next_state = None
        expValid = False
        if not self.future.running():
            # print("执行第一个动作")
            self.future = self.threadPool.submit(self.stepIpml,NextPoint)
            time.sleep(0.48)
            # print("第一个动作不计算奖励")
            stepLog = {}
        else:
            # print("等待当前动作完成")
            while not self.future.done():
                time.sleep(0.0001)
            # print("当前动作完成，开始执行下一个动作")
            self.future = self.threadPool.submit(self.stepIpml,NextPoint)
            time.sleep(0.49)#或加一个判断：即将到达目标点附近

            self.curRe = self.getRewardE()
            self.curRg = self.getRewardG()
            curRcvg = self.getRewardCoverageImprovement()
            base_reward = self.WE * self.curRe + self.WG * self.curRg + curRcvg
            stepLog = {
                't_gt':self.P_gt,
                'R_gt':self.P_gt_rot,
                't_est':self.P_est,
                'R_est':self.P_est_rot,
                'curRpe':self.curRpe,
                'Action':self.action,
                'curRe':self.curRe,
                'curRg':self.curRg,
                'curRcvg':curRcvg,
            }
            expValid = True

        # 获取下一个状态
        next_ftState,next_imuState = self.getCurState(NextPoint[4])
        if self.agent is not None and getattr(self.agent, 'StateNormalization', None) is not None:
            next_ftState = self.agent.StateNormalization(next_ftState)
        memberships = self.get_fuzzy_memberships()
        if expValid and self.op_cbrs is not None and self.last_ftState is not None and self.last_imuState is not None:
            self.reward, shape_info = self.op_cbrs.shape_reward(base_reward, self.last_ftState, self.last_imuState, next_ftState, next_imuState, float(self.action), memberships, self.recent_actions)
            stepLog.update(shape_info)
        elif expValid:
            self.reward = base_reward
        stepLog.update({'mu_T': memberships['mu_T'], 'mu_L': memberships['mu_L'], 'mu_W': memberships['mu_W'], 'mu_A': memberships['mu_A'], 'mu_cvg': memberships['mu_cvg'], 'TotalR': self.reward})
        self.last_ftState = next_ftState
        self.last_imuState = next_imuState
        if expValid:
            self.recent_actions.append(float(self.action))
            self.recent_actions = self.recent_actions[-8:]
        return next_ftState,next_imuState,self.reward,expValid,stepLog

    def stepOffline(self,NextPoint):
        ret_next_state = None
        expValid = False
        if not self.future.running():
            # print("执行第一个动作")
            self.future = self.threadPool.submit(self.stepIpml,NextPoint)
            time.sleep(0.48)
            # print("第一个动作不计算奖励")
            stepLog = {}
        else:
            # print("等待当前动作完成")
            while not self.future.done():
                time.sleep(0.0001)
            # print("当前动作完成，开始执行下一个动作")
            self.future = self.threadPool.submit(self.stepIpml,NextPoint)
            time.sleep(0.49)#或加一个判断：即将到达目标点附近
            self.curRe = self.getRewardE()
            self.curRg = self.getRewardG()
            curRcvg = self.getRewardCoverageImprovement()
            self.reward = self.WE * self.curRe + self.WG * self.curRg + curRcvg
            stepLog = {
                't_gt':self.P_gt,
                'R_gt':self.P_gt_rot,
                't_est':self.P_est,
                'R_est':self.P_est_rot,
                'curRpe':self.curRpe,
                'Action':self.action,
                'curRe':self.curRe,
                'curRg':self.curRg,
                'curRcvg':curRcvg,
                'TotalR':self.reward
            }
            expValid = True

        # 获取下一个状态
        next_ftState,next_imuState = self.getCurStateOffline()
        self.last_ftState = next_ftState
        self.last_imuState = next_imuState
        return next_ftState,next_imuState,self.reward,expValid,stepLog

    # uav training control
    def makeSureSlamIsLaunchSuccessfully(self):
        while self.isSlamLaunchSucc is False:
            self.env.slamLauncher()
            # print("lanuching")
            # time.sleep(30)
            time.sleep(10)
            if self.checkSlamIsLaunching() is True:
                time.sleep(1)
                return
            else:
                env.slamShutdown()
                # print("Launching fail and retry")
                time.sleep(2)

    def checkSlamIsLaunching(self):
        IsError = False
        for i in range(5):
            if self.checkError():
                IsError = True
            time.sleep(0.1)
        if IsError:
            return False
        else:
            return  True

    def waitUavHover(self):
        while self.env.uav.get_autopilot_state_name() != 'HOVER':
            time.sleep(0.01)

    def waitUavReadyForTask(self):
        while self.P_ref[2] < 6.9:
            time.sleep(0.01)
        time.sleep(2)

    def checkError(self):
        '''
        维护一个长度为5的窗口,用来记录slam的位姿估计,如果slam fail了,窗口内的位姿将不会更新
        如果窗口内的位姿都是一样的,说明slam fail了
        '''
        ret = True
        self.checkErrorBuffer[self.checkId%5]=self.P_est
        self.checkId+=1

        first = self.checkErrorBuffer[0][0]
        for i in self.checkErrorBuffer:
            if i[0] != first:
                ret = False
        return ret

    def OnlineCalibrPath(self,wd,DetectDeg):
        init_rate = rospy.Rate(1/2)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        # 前往任务起点
        self.env.uav.to_pose(0,0,4,0)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        self.waitUavHover()

        v = 0.4
        # set record flag True
        self.recordAlignTraj = True
        # online calibr
        self.env.uav.move_velo_time(-v,0,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(v,0,0,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(-v,0,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,-v,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,v,0,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,-v,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,-v,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,v,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,-v,0,0,0,6)
        init_rate.sleep()
        # set record flag False
        self.recordAlignTraj = False
        self.env.uav.move_velo_time(0,v,0,0,0,0,3)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()

        theta0=(270+DetectDeg)/180*math.pi
        x = (20+wd)*math.cos(theta0)
        y = (20+wd)*math.sin(theta0)

        self.env.uav.to_pose(x,y,7,DetectDeg)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        self.waitUavHover()

    def OnlineCalibrPath_2(self,wd,DetectDeg):
        init_rate = rospy.Rate(1/2)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        # 前往任务起点
        self.env.uav.to_pose(0,0,4,0)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        self.waitUavHover()

        v = 0.4
        # set record flag True
        self.recordAlignTraj = True
        # online calibr
        self.env.uav.move_velo_time(-v,0,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(v,0,0,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(-v,0,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,-v,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,v,0,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,-v,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,-v,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,v,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,-v,0,0,0,6)
        init_rate.sleep()
        # set record flag False
        self.recordAlignTraj = False

        self.env.uav.move_velo_time(0,0,v,0,0,0,4)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()

        theta0=(270+DetectDeg)/180*math.pi
        x = (20+wd)*math.cos(theta0)
        y = (20+wd)*math.sin(theta0) + 3+20+wd

        self.env.uav.to_pose(x,y,7,DetectDeg)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        self.waitUavHover()

    def reset_alignTraj(self):
        self.alignTraj_ref = []
        self.alignTraj_est = []

import torch
torch.cuda.is_available()


## Offline Inspection Trajectory Initialization

This section defines the offline inspection trajectory generation utilities used by the original workflow.


In [ ]:
# Task Path Initialized - Code block
def getCurYaw():
    x,y,z,w=tenv.P_gt_rot
    siny_cosp = 2* (z*w+x*y)
    cosy_cosp = 1 - 2 * (y*y + z*z)
    yaw = math.atan2(siny_cosp,cosy_cosp)/math.pi*180
    return yaw
def getEstYaw():
    x,y,z,w=tenv.P_est_rot
    siny_cosp = 2* (z*w+x*y)
    cosy_cosp = 1 - 2 * (y*y + z*z)
    yaw = math.atan2(siny_cosp,cosy_cosp)/math.pi*180
    return yaw

def moveToPoseAtOneSecond(goal):
    xt,yt,zt,wt,_,orient=goal
    x0,y0,z0=tenv.P_gt
    w0 = getCurYaw()
    vx=xt-x0
    vy=yt-y0
    vz=zt-z0
    vw=(wt-w0)*1/60
    tenv.env.uav.move_velo_time(vx,vy,vz,0,0,vw,1)

def SetSectorDeg(DetectionDeg:[],wallDist):#设置覆盖扇区的角度
    colDeg = []
    deg = 270#初始角度
    # ddeg = 4.3#设置横移步长，影响着巡检轨迹的列数
    R=20 + wallDist
    ddeg = 4.3
    for i in range(int(DetectionDeg[1]-DetectionDeg[0]/4.3)):
        theta = (deg+i*ddeg)/180*math.pi
        x = R*math.cos(theta)
        y = R*math.sin(theta)+ R
        colDeg.append(np.array([x,y,i*ddeg],dtype=float))
    return colDeg

def SetSectorDeg_2(DetectionDeg:[],wallDist):#设置覆盖扇区的角度
    colDeg = []
    deg = 270 + DetectionDeg[0] #初始角度

    # ddeg = 4.3#设置横移步长，影响着巡检轨迹的列数
    R=20 + wallDist
    ddeg = 4.3

    theta0=(deg)/180*math.pi
    xb = -R*math.cos(theta0)
    yb = -R*math.sin(theta0)

    for i in range(int((DetectionDeg[1]-DetectionDeg[0])/4.3)):
        theta = (deg+i*ddeg)/180*math.pi
        x = R*math.cos(theta)+ xb
        y = R*math.sin(theta)+ yb
        colDeg.append(np.array([x,y,i*ddeg],dtype=float))
    return colDeg

def SetSectorHeightRange(DetectingSector,high):#设置检查区域的高度
    TurningPointSet = []
    l=0
    h=high
    for i in range(0,len(DetectingSector)):
        x,y,w=DetectingSector[i]
        TurningPointSet.append((x,y,l,w,1,0))
        TurningPointSet.append((x,y,h,w,1,0))
        h,l = l,h#采用蛇形走位
    return TurningPointSet
def GenRoutePointSet(OriginPoint, TurningPointSet):#给定原点与轨迹，生成航线，并且设在每一个航迹点的方向
    RoutePointSet = []
    loop = [0, 1, 2, 1]
    x0, y0, z0, w0, _, _ = OriginPoint
    x, y, z, w, _, _ = TurningPointSet[0]
    RoutePointSet.append((x + x0, y + y0, z + z0, w + w0, 1, 0))
    for i in range(1, len(TurningPointSet)):
        x, y, z, w, _, _ = TurningPointSet[i]
        RoutePointSet.append((x + x0, y + y0, z + z0, w + w0, 1, loop[(i - 1) % 4]))
    return RoutePointSet

def CoveragePathPlanner(OriginPoint,DetectionDeg,high,wallDist):
    """
    新增: 调整与墙面的距离R
    巡检区域可视为圆柱体侧表面的某段弧形曲面，用左下角的起点，弧形的扇形大小，曲面的高度来进行定义.
    令U为无人机的坐标系原点
    slam的位置估计是在U坐标系下的
    因此航线的规划应该在U坐标系下直接规划。
    :param OriginPoint: 覆盖路径的起点
    :param DetectionDeg: 覆盖区域的扇形角度
    :param high: 覆盖区域的高度
    :param wallDist: 抵近测量距离
    :return: CoveragePointSet 巡检路径的拐点集
    :rtype: (x,y,z,w,orientation) orientation: 0 向上飞行 // 1 向下飞行 // 2 向右飞行
    """
    PUO = np.array(list(OriginPoint)[0:4])

    DetectingSector= SetSectorDeg(DetectionDeg,wallDist)
    x0,y0,z0,w0 = PUO[0:4]
    CoveragePointSet = []#PUP_Set
    l=0
    h=high
    for i in range(0,len(DetectingSector)):
        x,y,w=DetectingSector[i]
        CoveragePointSet.append([x0+x,y0+y,z0+l,w0+w,0])
        CoveragePointSet.append([x0+x,y0+y,z0+h,w0+w,0])
        h,l = l,h#采用蛇形走位

    loop = [0, 2, 1, 2]
    for i in range(1, len(CoveragePointSet)):
        CoveragePointSet[i][4] = loop[(i - 1) % 4]
    return CoveragePointSet

def CoveragePathPlanner_2(OriginPoint,DetectionDeg,high,wallDist):
    """
    新增: 调整与墙面的距离R
    巡检区域可视为圆柱体侧表面的某段弧形曲面，用左下角的起点，弧形的扇形大小，曲面的高度来进行定义.
    令U为无人机的坐标系原点
    slam的位置估计是在U坐标系下的
    因此航线的规划应该在U坐标系下直接规划。
    :param OriginPoint: 覆盖路径的起点
    :param DetectionDeg: 覆盖区域的扇形角度
    :param high: 覆盖区域的高度
    :param wallDist: 抵近测量距离
    :return: CoveragePointSet 巡检路径的拐点集
    :rtype: (x,y,z,w,orientation) orientation: 0 向上飞行 // 1 向下飞行 // 2 向右飞行
    """
    PUO = np.array(list(OriginPoint)[0:4])

    DetectingSector= SetSectorDeg_2(DetectionDeg,wallDist)
    x0,y0,z0,w0 = PUO[0:4]
    CoveragePointSet = []#PUP_Set
    l=0
    h=high
    for i in range(0,len(DetectingSector)):
        x,y,w=DetectingSector[i]
        CoveragePointSet.append([x0+x,y0+y,z0+l,w0+w,0])
        CoveragePointSet.append([x0+x,y0+y,z0+h,w0+w,0])
        h,l = l,h#采用蛇形走位

    loop = [0, 2, 1, 2]
    for i in range(1, len(CoveragePointSet)):
        CoveragePointSet[i][4] = loop[(i - 1) % 4]
    return CoveragePointSet

def DiscreteTraj(Traj,discreteDegree):#将航线离散化
    OutputTraj=[]
    n=len(Traj)
    for i in range(n-1):
        x0,y0,z0,w0,_,_=Traj[i]
        xt,yt,zt,wt,_,orient=Traj[i+1]
        norm = np.linalg.norm((xt-x0,yt-y0,zt-z0))
        N = math.floor(norm/discreteDegree)
        dx = (xt-x0)/N
        dy = (yt-y0)/N
        dz = (zt-z0)/N
        dw = (wt-w0)/N
        for j in range(N-1):
            OutputTraj.append((
                x0+(j+1)*dx,y0+(j+1)*dy,z0+(j+1)*dz,w0+(j+1)*dw,0,orient
            ))
        OutputTraj.append((
                x0+N*dx,y0+N*dy,z0+N*dz,w0+N*dw,1,orient
            ))
    return OutputTraj

def FollowByLOS(Traj,sight):#LOS算法轨迹跟踪
    n = len(Traj)
    x0,y0,z0 = tenv.P_gt#后期加入yaw
    for i in range(n):
        xt,yt,zt,_,isTp,_ = Traj[i]
        dist = np.linalg.norm((xt-x0,yt-y0,zt-z0))
        if isTp==1:#如果遍历到转角，飞往转角并悬停
            tenv.waitUavHover()
            moveToPoseAtOneSecond(Traj[i])
            tenv.waitUavHover()
            continue
        # 当前轨迹点不是转角，用LOS算法选择一个目标点
        if (dist<sight):
            continue
        else :
            # print(Traj[i-1])
            moveToPoseAtOneSecond(Traj[i-1])
#             x0,y0,z0 = tenv.P_gtsource devel/setup.bash
# source devel/setup.bash
            x0,y0,z0 = tenv.P_gtsource
            # source devel/setup.bash
import time
def setOriginTUMPoint():
    sum_x=0
    sum_y=0
    sum_z=0
    sum_q0=0
    sum_q1=0
    sum_q2=0
    sum_q3=0
    for i in range(20):
        x,y,z=tenv.P_est
        q0,q1,q2,q3=tenv.P_est_rot
        sum_x+=x
        sum_y+=y
        sum_z+=z
        sum_q0+=q0
        sum_q1+=q1
        sum_q2+=q2
        sum_q3+=q3
        time.sleep(0.05)
    return (sum_x/20,sum_y/20,sum_z/20,sum_q0/20,sum_q1/20,sum_q2/20,sum_q3/20,1,0)

def setOriginPoint():
    sum_x=0
    sum_y=0
    sum_z=0
    sum_w=0
    for i in range(20):
        x,y,z=tenv.P_est
        w=getEstYaw()
        sum_x+=x
        sum_y+=y
        sum_z+=z
        sum_w+=w
        time.sleep(0.05)
    return (sum_x/20,sum_y/20,sum_z/20,sum_w/20,1,0)

def GetTaskStartPoint():
    sum_x=0
    sum_y=0
    sum_z=0
    sum_w=0
    for i in range(20):
        x,y,z=tenv.P_est
        w=getEstYaw()
        sum_x+=x
        sum_y+=y
        sum_z+=z
        sum_w+=w
        time.sleep(0.05)
    return (sum_x/20,sum_y/20,sum_z/20,sum_w/20,0)

def moveToEstPoseAtOneSecond(goal):
    xt,yt,zt,wt,_,orient=goal
    x0,y0,z0=tenv.P_est
    w0 = getEstYaw()
    # print(f'{x0,y0,z0}')
    # print(f'w0:{w0},wt:{wt}')
    vx=xt-x0
    vy=yt-y0
    vz=zt-z0
    vw=(wt-w0)*1/60 #这里有区别 负数
    tenv.env.uav.move_velo_time(vx,vy,vz,0,0,vw,1)
    time.sleep(0.1)

def FollowingTrajBySlam(Traj,sight):#LOS算法轨迹跟踪
    n = len(Traj)
    x0,y0,z0 = tenv.P_est
    for i in range(n):
        xt,yt,zt,_,isTp,_ = Traj[i]
        dist = np.linalg.norm((xt-x0,yt-y0,zt-z0))
        if isTp==1:#如果遍历到转角，飞往转角并悬停
            print(f'arrived TP:{Traj[i]}')
            print(f'dt = {np.linalg.norm(tenv.P_est - tenv.P_gt)}')
            tenv.waitUavHover()
            moveToEstPoseAtOneSecond(Traj[i])
            tenv.waitUavHover()
            continue
        # 当前轨迹点不是转角，用LOS算法选择一个目标点
        if (dist<sight):
            continue
        else :
            moveToEstPoseAtOneSecond(Traj[i-1])
            x0,y0,z0 = tenv.P_est
def printP_est():
    print(f'X = {tenv.P_est[0]}')
    print(f'Y = {tenv.P_est[1]}')
    print(f'Z = {tenv.P_est[2]}')
    print(f'Yaw = {getEstYaw()}')
def printP_gt():
    print(f'X = {tenv.P_gt[0]}')
    print(f'Y = {tenv.P_gt[1]}')
    print(f'Z = {tenv.P_gt[2]}')
    print(f'Yaw = {getCurYaw()}')

def moveToEstPoseAtNSecond(goal,N):
    xt,yt,zt,wt,_,orient=goal
    x0,y0,z0=tenv.P_est
    w0 = getEstYaw()
    vx=(xt-x0)/N
    vy=(yt-y0)/N
    vz=(zt-z0)/N
    vw=((wt-w0)*(1/60))/N #这里有区别 负数
    tenv.env.uav.move_velo_time(vx,vy,vz,0,0,vw,N)
    time.sleep(0.1)


## Inspection Path Following and Tracking Utilities

This section contains path-following, next-point computation, and trajectory tracking helpers.


In [ ]:
# Path Following algorithm - Code block
def lineseg_dist(P, A, B):# 计算 P点 到  A B 所确定的直线的最短距离
    return np.linalg.norm(np.cross(B-A,P-A))/np.linalg.norm(B-A)

def CalcNextPoint(CurPosition,PrevTargetPoint,CurTargetPoint,ViewDist):
    ## 计算下一个目标点
    # 变量见实验笔记
    # CurPosition:C PrevPosition:A  TarPosition:B
    PrevPosition = np.array(list(PrevTargetPoint[:3]))
    TarPosition = np.array(list(CurTargetPoint[:3]))
    orient = int((PrevPosition[2] - TarPosition[2])>0) # 向上飞行：0 向下飞行：1
    Za = PrevPosition[2]
    Zc = CurPosition[2]

    v = ViewDist
    a = np.linalg.norm(CurPosition - PrevPosition)
    d = lineseg_dist(CurPosition,PrevPosition,TarPosition)

    tmp = a*a - d*d
    if tmp<0:
        tmp = 0
    b = math.sqrt(tmp)

    tmp = v*v - d*d
    if tmp<0:
        tmp = 0
    c = math.sqrt(tmp)
    if orient == 0:# 向上飞行
        if Za < Zc:# C 在 A B 之间
            NextPoint = np.array([PrevTargetPoint[0],PrevTargetPoint[1],PrevTargetPoint[2]+b+c,PrevTargetPoint[3],orient])
        else:# C 在 A B 之外
            NextPoint = np.array([PrevTargetPoint[0],PrevTargetPoint[1],PrevTargetPoint[2]+c-b,PrevTargetPoint[3],orient])
    else: # 向下飞行
        if Za > Zc:# C 在 A B 之间
            NextPoint = np.array([PrevTargetPoint[0],PrevTargetPoint[1],PrevTargetPoint[2]-b-c,PrevTargetPoint[3],orient])
        else:# C 在 A B 之外
            NextPoint = np.array([PrevTargetPoint[0],PrevTargetPoint[1],PrevTargetPoint[2]-c+b,PrevTargetPoint[3],orient])
    return NextPoint

def CarefullyMoveToPoint(point,t):
    xt,yt,zt,wt,orient=point
    x0,y0,z0=tenv.P_est
    w0 = getEstYaw()
    vx=(xt-x0)/t
    vy=(yt-y0)/t
    vz=(zt-z0)/t
    vw=((wt-w0)*1/60)/t
    tenv.env.uav.move_velo_time(vx,vy,vz,0,0,vw,t)

def moveDeltaPoseAtTimeT(x,y,z,h,t):
    vx=x/t
    vy=y/t
    vz=z/t
    yawh=h*1/60/t
    tenv.env.uav.move_velo_time(vx,vy,vz,0,0,yawh,t)

def CarefullyFlyAround(A,B,O):
    # 给定两个点，生成这两个点对应在园上的弧的 离散的点
    initDeg = 270 + A[3] - O[3]
    initPos = O
    ddeg = 0.5
    pointNum = int((B[3]-A[3])/0.5)
    R=21
    arc = []
    for i in range(1,pointNum):
        theta = (initDeg+i*ddeg)/180*math.pi
        x = R*math.cos(theta)
        y = R*math.sin(theta)+21
        arc.append(np.array([x,y,0],dtype=float)) # (0,0,10)

    for i in range(len(arc)):
        arc[i][0] = arc[i][0] + initPos[0]
        arc[i][1] = arc[i][1] + initPos[1]

    preX = A[0]
    preY = A[1]
    for p in arc:
        # 计算 delta x delta y
        dx = p[0]-preX
        dy = p[1]-preY
        preX = p[0]
        preY = p[1]
        moveDeltaPoseAtTimeT(dx,dy,0,0.575,1)

def CarefullyFlyAround_2(DetectDeg,A,B,O):
    # 给定两个点，生成这两个点对应在园上的弧的 离散的点
    initDeg = 270 + DetectDeg[0] + A[3] - O[3]
    initPos = A
    ddeg = 0.5
    pointNum = int((B[3]-A[3])/0.5)
    R=21
    arc = []

    theta0=(initDeg)/180*math.pi
    xb = -R*math.cos(theta0)
    yb = -R*math.sin(theta0)
    # 以
    for i in range(1,pointNum):
        theta = (initDeg+i*ddeg)/180*math.pi
        x = R*math.cos(theta)+ xb
        y = R*math.sin(theta)+ yb

        arc.append(np.array([x,y,0],dtype=float)) # (0,0,10)

    for i in range(len(arc)):
        arc[i][0] = arc[i][0] + initPos[0]
        arc[i][1] = arc[i][1] + initPos[1]

    preX = A[0]
    preY = A[1]
    for p in arc:
        # 计算 delta x delta y
        dx = p[0]-preX
        dy = p[1]-preY
        preX = p[0]
        preY = p[1]
        moveDeltaPoseAtTimeT(dx,dy,0,0.575,1)

def CarefullyTurnAround(CoveragePointSet,CurTargetIdx):
    O = CoveragePointSet[0]
    A = CoveragePointSet[CurTargetIdx]
    B = CoveragePointSet[CurTargetIdx+1]
    tenv.waitUavHover()
    CarefullyMoveToPoint(A,2)
    tenv.waitUavHover()
    CarefullyFlyAround(A,B,O)
    tenv.waitUavHover()
    CarefullyMoveToPoint(B,2)
    tenv.waitUavHover()

def CarefullyTurnAround_2(DetectDeg,CoveragePointSet,CurTargetIdx):
    O = CoveragePointSet[0]
    A = CoveragePointSet[CurTargetIdx]
    B = CoveragePointSet[CurTargetIdx+1]
    tenv.waitUavHover()
    CarefullyMoveToPoint(A,2)
    tenv.waitUavHover()
    CarefullyFlyAround_2(DetectDeg,A,B,O)
    tenv.waitUavHover()
    CarefullyMoveToPoint(B,2)
    tenv.waitUavHover()

def checkIsDeviated(CoveragePointSet,PrevTargetIdx,CurTargetIdx):
    if PrevTargetIdx>=len(CoveragePointSet):
        return False
    PrevTargetPoint=CoveragePointSet[PrevTargetIdx]
    CurTargetPoint=CoveragePointSet[CurTargetIdx]
    PrevPosition = np.array(list(PrevTargetPoint[:3]))
    TarPosition = np.array(list(CurTargetPoint[:3]))
    d = lineseg_dist(tenv.P_est,PrevPosition,TarPosition)
    print(f'与GT航线的最短距离为{d}')
    return d>0.5

def calcDeviatedDist(CoveragePointSet,PrevTargetIdx,CurTargetIdx):
    if PrevTargetIdx>=len(CoveragePointSet):
        return False
    PrevTargetPoint=CoveragePointSet[PrevTargetIdx]
    CurTargetPoint=CoveragePointSet[CurTargetIdx]
    PrevPosition = np.array(list(PrevTargetPoint[:3]))
    TarPosition = np.array(list(CurTargetPoint[:3]))
    d = lineseg_dist(tenv.P_est,PrevPosition,TarPosition)
    # print(f'与GT航线的最短距离为{d}')
    return d

# 在线校正
import numpy as np
import typing

UmeyamaResult = typing.Tuple[np.ndarray, np.ndarray, float]

def umeyama_alignment(x: np.ndarray, y: np.ndarray, with_scale: bool = False) -> UmeyamaResult:
    """
    Computes the least squares solution parameters of an Sim(m) matrix
    that minimizes the distance between a set of registered points.
    Umeyama, Shinji: Least-squares estimation of transformation parameters
                     between two point patterns. IEEE PAMI, 1991
    :param x: mxn matrix of points, m = dimension, n = nr. of data points
    :param y: mxn matrix of points, m = dimension, n = nr. of data points
    :param with_scale: set to True to align also the scale (default: 1.0 scale)
    :return: r, t, c - rotation matrix, translation vector and scale factor
    """
    # m = dimension, n = nr. of data points
    m, n = x.shape

    # means, eq. 34 and 35
    mean_x = x.mean(axis=1)
    mean_y = y.mean(axis=1)

    # variance, eq. 36
    # "transpose" for column subtraction
    sigma_x = 1.0 / n * (np.linalg.norm(x - mean_x[:, np.newaxis])**2)

    # covariance matrix, eq. 38
    outer_sum = np.zeros((m, m))
    for i in range(n):
        outer_sum += np.outer((y[:, i] - mean_y), (x[:, i] - mean_x))
    cov_xy = np.multiply(1.0 / n, outer_sum)

    # SVD (text betw. eq. 38 and 39)
    u, d, v = np.linalg.svd(cov_xy)
    if np.count_nonzero(d > np.finfo(d.dtype).eps) < m - 1:
        print("Degenerate covariance rank, Umeyama alignment is not possible")

    # S matrix, eq. 43
    s = np.eye(m)
    if np.linalg.det(u) * np.linalg.det(v) < 0.0:
        # Ensure a RHS coordinate system (Kabsch algorithm).
        s[m - 1, m - 1] = -1

    # rotation, eq. 40
    r = u.dot(s).dot(v)

    # scale & translation, eq. 42 and 41
    c = 1 / sigma_x * np.trace(np.diag(d).dot(s)) if with_scale else 1.0
    t = mean_y - np.multiply(c, r.dot(mean_x))

    return r, t, c
def writeOnlineAlignTxt():
    x = tenv.alignTraj_est
    y = tenv.alignTraj_ref
    x = np.asarray(x).transpose()
    y = np.asarray(y).transpose()
    R,t,s = umeyama_alignment(x,y)
    with open("/home/dbq/dbq/DRL_SLAM/slam/exp/fm_orbslam3/stereoInertial/OnlineAlign.txt",'w') as f:
        for r in R:
            for c in r:
                f.write(f'{float(c)}\n')
        for r in t:
            f.write(f'{float(r)}\n')

def calcAlignT():
    x = tenv.alignTraj_est
    y = tenv.alignTraj_ref
    x = np.asarray(x).transpose()
    y = np.asarray(y).transpose()
    R,t,s = umeyama_alignment(x,y)
    return R,t

def ReAlignSLAM():
    intmsg = std_msgs.Int8()
    intmsg.data= 0
    tenv.ReAlignFlagPub.publish(intmsg)

# PPO Agent Code Block
# 这一个代码块定义了 Agent 的网络结构、训练超参数、网络更新函数等功能
import os
import time
import numpy as np
import math
import torch
import torch.nn as nn
from torch import Tensor
from torch.distributions.normal import Normal
from torch.nn import functional as F

def build_mlp(dims: [int]) -> nn.Sequential:  # MLP (MultiLayer Perceptron)
    net_list = []
    for i in range(len(dims) - 1):
        net_list.extend([nn.Linear(dims[i], dims[i + 1]), nn.Tanh()])# Trick 10 : Tanh activation func
    del net_list[-1]  # remove the activation of output layer
    return nn.Sequential(*net_list)

class ActorCnn(nn.Module):
    def __init__(self, dims: [int], state_dim: int, action_dim: int,action_std_log_init:float):
        super().__init__()
        # 动作方差的对数
        self.action_std_log = nn.Parameter(torch.tensor(action_std_log_init), requires_grad=True)

        self.conv1 = nn.Sequential(         # input shape (1, 48, 72)
            nn.Conv2d(1,16,8,4,2),
            nn.ReLU(),                      # output shape (8, 12, 18)
            nn.Conv2d(16,32,4,2,0),
            nn.ReLU(),                      # output shape (32, 5, 8)
            nn.MaxPool2d(2),                # output shape (32, 2, 4)
        )

        self.conv2 = nn.Sequential(         # input shape (32, 2, 4)
            nn.Conv2d(32, 64, 2, 1, 0),     # output shape (64, 1, 3)
            nn.ReLU(),                      # activation
        )

        self.fc1 = nn.Linear(64 * 1 * 3, state_dim - 6)   #  fully connected layer 1

        self.fc2 = build_mlp(dims=[state_dim, *dims, action_dim]) #  fully connected layer 1

    def forward(self, ftState: Tensor, imuState: Tensor) -> Tensor: # [-1, 1]
        x = self.conv1(ftState)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)# flatten ft map
        x = F.tanh(self.fc1(x))# fully connected layer 1
        x = torch.cat((x,imuState),dim = -1)# cat dynamic info
        x = F.tanh(self.fc2(x))# fully connected layer 2
        return x

    # 获取动作与动作的概论值
    def get_action(self, ftState: Tensor, imuState: Tensor) -> (Tensor, Tensor):  # for exploration
        # 获得动作的分布
        action_avg = self.forward(ftState,imuState) # S -> [-1,1]
        action_std = self.action_std_log.exp()
        # 采样
        dist = Normal(action_avg, action_std)#取决于std，如果std取值不当，会使值映射在[-1,1]区间外的范围，如std=0,[-4,4]
        action = dist.sample()# 采样动作
        logprob = dist.log_prob(action)# 计算采样到该动作的概论
        return action, logprob

    def get_logprob_entropy(self, ftState: Tensor, imuState: Tensor, action: Tensor) -> (Tensor, Tensor):
        action_avg = self.forward(ftState,imuState)
        action_std = self.action_std_log.exp()

        dist = Normal(action_avg, action_std)
        logprob = dist.log_prob(action)
        entropy = dist.entropy()
        return logprob, entropy

class CriticCnn(nn.Module):
    def __init__(self, dims: [int], state_dim: int, _action_dim: int):
        super().__init__()
        self.conv1 = nn.Sequential(         # input shape (1, 48, 72)
            nn.Conv2d(1,16,8,4,2),
            nn.ReLU(),                      # output shape (8, 12, 18)
            nn.Conv2d(16,32,4,2,0),
            nn.ReLU(),                      # output shape (32, 5, 8)
            nn.MaxPool2d(2),                # output shape (32, 2, 4)
        )
        self.conv2 = nn.Sequential(         # input shape (32, 2, 4)
            nn.Conv2d(32, 64, 2, 1, 0),     # output shape (64, 1, 3)
            nn.ReLU(),                      # activation
        )
        self.fc1 = nn.Linear(64 * 1 * 3, state_dim - 6)   #  fully connected layer
        self.fc2 = build_mlp(dims=[state_dim, *dims, 1])

    def forward(self, ftState: Tensor, imuState: Tensor) -> Tensor:
        x = self.conv1(ftState)                 # conv 2d layer 1
        x = self.conv2(x)                       # conv 2d layer 2
        x = x.view(x.size(0), -1)               # flatten ft map
        x = self.fc1(x).tanh()              # fully connected layer 1
        x = torch.cat((x,imuState),dim = -1)    # cat dynamic info
        x = self.fc2(x)                         # fully connected layer 2
        return x

class ActorFc(nn.Module):
    def __init__(self, dims: [int], state_dim: int, action_dim: int,action_std_log_init:float):
        super().__init__()
        # 动作方差的对数
        self.action_std_log = nn.Parameter(torch.tensor(float(action_std_log_init), dtype=torch.float32))

        # 创建全链接网络
        self.net = build_mlp(dims=[state_dim, *dims, action_dim])

    def forward(self, ftState: Tensor, imuState: Tensor) -> Tensor:
        x = torch.cat((ftState,imuState),dim = -1)# cat dynamic info
        return self.net(x).tanh()  # action.tanh()

    # 获取动作与动作的概论值
    def get_action(self, ftState: Tensor, imuState: Tensor) -> Tensor:  # for exploration
        # 获得动作的分布
        action_avg = self.forward(ftState,imuState)
        action_std = self.action_std_log.exp()
        # 采样
        dist = Normal(action_avg, action_std)
        action = dist.sample()# 采样动作
        logprob = dist.log_prob(action)# 计算采样到该动作的概论
        return action, logprob, action_avg

    def get_logprob_entropy(self, ftState: Tensor, imuState: Tensor, action: Tensor) -> (Tensor, Tensor):
        action_avg = self.forward(ftState,imuState)
        action_std = self.action_std_log.exp()

        dist = Normal(action_avg, action_std)
        logprob = dist.log_prob(action)
        entropy = dist.entropy()
        return logprob, entropy

    def set_action_std(self, new_action_std):
            self.action_std_log.data.fill_(float(new_action_std))

    @staticmethod
    def convert_action_for_env(action: Tensor) -> Tensor:
        return action.tanh()

class CriticFc(nn.Module):
    def __init__(self, dims: [int], state_dim: int, _action_dim: int):
        super().__init__()
        self.net = build_mlp(dims=[state_dim, *dims, 1])

    def forward(self, ftState: Tensor, imuState: Tensor) -> Tensor:
        x = torch.cat((ftState,imuState),dim = -1)# cat dynamic info
        return self.net(x)

class PPO_Config:
    def __init__(self):
        # 网络结构
        self.NetArc = 'fc'  # fc / cnn
        if self.NetArc == 'fc':
            # 384 pooled visual features + 10 dynamic/fuzzy features
            self.state_dim = 394
            self.net_dims = [256,128,64]
        else:
            self.state_dim = 132
            self.net_dims = [128,64]

        self.action_dim = 1
        self.action_std_log = -1.5
        self.action_std_decay_rate = 0.1
        self.min_action_std = -4.0
        self.sycnUd = False

        self.gpu_id = 0
        self.device = torch.device(f"cuda:{self.gpu_id}" if (torch.cuda.is_available() and (self.gpu_id >= 0)) else "cpu")

        # Paper-aligned PPO defaults
        self.gamma = 0.95
        self.reward_scale = 1.0
        self.ratio_clip = 0.25
        self.lambda_gae_adv = 0.98
        self.lambda_entropy = 0.01
        self.lambda_entropy = torch.tensor(self.lambda_entropy, dtype=torch.float32, device=self.device)

        self.batch_size = int(512)
        self.horizon_len = int(1024)
        self.repeat_times = 16.0

        self.gpu_id = int(0)
        self.usedLRD = False
        self.update_target_time = 100
        self.max_total_steps = self.update_target_time * self.horizon_len * self.repeat_times
        self.learning_rate = 6e-5
        self.PhiNetPath = 'no path'

class RunningMeanStd:
    # Dynamically calculate mean and std
    def __init__(self, shape):  # shape:the dimension of input data
        self.n = 0
        self.mean = 0.0
        self.S = 0.0
        self.std = math.sqrt(self.S)
        self.shape = shape

    def update(self, x):
        self.n += 1
        if self.n == 1:
            self.mean = x.mean()
            self.std = x.std()
        else:
            x_CurMean = x.mean()
            old_mean = self.mean
            self.mean = old_mean + (x_CurMean - old_mean) / (self.n * self.shape)
            self.S = self.S + (x_CurMean - old_mean) * (x_CurMean - self.mean)
            self.std = math.sqrt(self.S / self.n )

class Normalization:
    def __init__(self,r,c):
        self.running_ms = RunningMeanStd(shape=r*c)
        self.shape = r*c
        self.r = r
        self.c = c
        self.update = True
    def __call__(self, x):
        # Whether to update the mean and std,during the evaluating,update=Flase
        if self.update:
            self.running_ms.update(x)
        device = x.device
        mu = torch.ones((self.r,self.c), device=device) * self.running_ms.mean
        std = torch.ones((self.r,self.c), device=device) * max(self.running_ms.std, 1e-6)
        x = (x-mu)/std
        return x

class AgentPPO():
    def __init__(self, cfg: PPO_Config = PPO_Config()):
        '''获取参数'''
        self.net_dims = cfg.net_dims
        self.state_dim = cfg.state_dim
        self.action_dim = cfg.action_dim

        self.action_std_log = cfg.action_std_log
        self.action_std_decay_rate = cfg.action_std_decay_rate
        self.min_action_std = cfg.min_action_std

        self.gamma = cfg.gamma
        self.reward_scale = cfg.reward_scale
        self.ratio_clip = cfg.ratio_clip
        self.lambda_gae_adv = cfg.lambda_gae_adv
        self.lambda_entropy =  cfg.lambda_entropy

        self.learning_rate = cfg.learning_rate
        self.horizon_len = cfg.horizon_len
        self.repeat_times = cfg.repeat_times
        self.batch_size = cfg.batch_size

        self.NetArc = cfg.NetArc
        # self.PhiNetPath = cfg.PhiNetPath

        self.last_state = None  # save the last state of the trajectory for training. `last_state.shape == (state_dim)`
        self.device = torch.device(f"cuda:{cfg.gpu_id}" if (torch.cuda.is_available() and (cfg.gpu_id >= 0)) else "cpu")

        # 定义actor与critic
        if self.NetArc == 'fc':
            self.act = ActorFc(self.net_dims, self.state_dim, self.action_dim,self.action_std_log).to(self.device)
            self.cri = CriticFc(self.net_dims, self.state_dim, self.action_dim).to(self.device)
        # else:
        #     self.act = ActorCnn(self.net_dims, self.state_dim, self.action_dim,self.action_std_log).to(self.device)
        #     self.cri = CriticCnn(self.net_dims, self.state_dim, self.action_dim).to(self.device)

        # 定义优化器 # Trick 9 : Adam Optimizer Epsillon Parameter
        self.act_optimizer = torch.optim.Adam(self.act.parameters(), self.learning_rate,eps=1e-5)
        self.cri_optimizer = torch.optim.Adam(self.cri.parameters(), self.learning_rate,eps=1e-5)
        # 定义两个网络参数距离的标准
        self.criterion = torch.nn.SmoothL1Loss()
        # Trick 2 : State Normaliztion
        if self.NetArc == 'fc':
            self.StateNormalization = Normalization(r=1,c=384)# only normalize vision feature
        else:
            self.StateNormalization = Normalization(r=48,c=72)# only normalize vision feature

        # Trick 6 : Learning Rate Decay
        self.usedLRD = cfg.usedLRD
        self.total_steps = 0
        self.update_target_time = cfg.update_target_time
        self.max_total_steps = cfg.max_total_steps

        # 是否同步更新actor和critic
        self.sycnUd = cfg.sycnUd

        # # 定义Phi
        # self.Phi = CriticPPO(self.net_dims, self.state_dim, self.action_dim).to(self.device)
        # self.Phi.load_state_dict(torch.load(self.PhiNetPath, map_location=lambda storage, loc: storage))

        # critic loss
        self.criticLoss = []

    def bind_env(self, test_env: TestEnv):
        test_env.bind_agent(self)

    @torch.no_grad()
    def select_action(self, ft_state: Tensor, imu_state: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
        action, logprob, action_avg = self.act.get_action(ft_state, imu_state)
        return action, logprob, action_avg

    def save(self, actor_path: str, critic_path: Optional[str] = None) -> None:
        torch.save(self.act.state_dict(), actor_path)
        if critic_path is not None:
            torch.save(self.cri.state_dict(), critic_path)

    def load(self, actor_path: str, critic_path: Optional[str] = None, strict: bool = True) -> None:
        self.act.load_state_dict(torch.load(actor_path, map_location=self.device), strict=strict)
        if critic_path is not None:
            self.cri.load_state_dict(torch.load(critic_path, map_location=self.device), strict=strict)

    def lr_decay(self,total_steps):
        lr_now = self.learning_rate * (1 - total_steps/self.max_total_steps)
        self.act_optimizer.param_groups[0]['lr'] = lr_now
        self.cri_optimizer.param_groups[0]['lr'] = lr_now

    def set_action_std(self, new_action_std):
        self.action_std_log = new_action_std
        self.act.set_action_std(new_action_std)

    def decay_action_std(self):
        self.action_std_log = self.action_std_log - self.action_std_decay_rate
        self.action_std_log = round(self.action_std_log, 4)
        if (self.action_std_log <= self.min_action_std):
            self.action_std_log = self.min_action_std
        self.set_action_std(self.action_std_log)

    def update_net(self, buffer) -> [float]:
        # 计算Value，该Value是用behavior critic 算出来的
        with torch.no_grad():
            ftStates, imuStates, actions, logprobs, rewards, undones = buffer
            buffer_size = ftStates.shape[0]
            '''get advantages reward_sums'''
            bs = 2 ** 10  # set a smaller 'batch_size' when out of GPU memory.
            values = [self.cri(ftStates[i:i + bs],imuStates[i:i + bs]) for i in range(0, buffer_size, bs)]
            values = torch.cat(values, dim=0).squeeze(1)

            advantages = self.get_advantages(rewards, undones, values)  # 计算GAE优势函数值

            reward_sums = advantages + values  # reward_sums = reward + next_value = Q
            del rewards, undones, values

            # Trick 1: Advantage Normalization ：： 使用GAE计算完一个batch中的advantage后，计算整个batch中所有advantage的mean和std，然后减均值再除以标准差。
            advantages = (advantages - advantages.mean()) / (advantages.std(dim=0) + 1e-5)
        assert logprobs.shape == advantages.shape == reward_sums.shape == (buffer_size,)

        '''update network'''
        obj_critics = 0.0
        obj_actors = 0.0

        update_times = int(buffer_size * self.repeat_times / self.batch_size)# 使用这些经验更新很多次
        assert update_times >= 1
        for repId in range(update_times):
            print(f'repId:{repId}')
            # 随机获取一个batch的经验
            indices = torch.randint(buffer_size, size=(self.batch_size,), requires_grad=False)
            ftState = ftStates[indices]
            imuState = imuStates[indices]
            action = actions[indices]
            logprob = logprobs[indices]
            advantage = advantages[indices]
            reward_sum = reward_sums[indices]

            # 更新Learning Rate
            if self.usedLRD:
                self.lr_decay(self.total_steps)

            # 先更新 critic
            value = self.cri(ftState,imuState).squeeze(1)  # V值 ，reward_sum 为 Q 值
            obj_critic = self.criterion(value, reward_sum)# 用smoothL1计算 Q - V ，即TD-error
            self.optimizer_update(self.cri_optimizer, obj_critic)# ctitic的更新用TD-error作为loss，自举

            # # 每一轮只更新4次actor
            # if self.sycnUd or repId < 3 :

            # 这里的act是更新后的 behavior policy, state 和 action 是原来的behavior policy收集的
            new_logprob, obj_entropy = self.act.get_logprob_entropy(ftState, imuState, action)# 输出对数概率值 策略的熵
            ratio = (new_logprob - logprob.detach()).exp()# 重要性采样：更新后的分布的采样概率/行为分布的采样概率
            surrogate1 = advantage * ratio
            surrogate2 = advantage * ratio.clamp(1 - self.ratio_clip, 1 + self.ratio_clip)
            obj_surrogate = torch.min(surrogate1, surrogate2).mean()# PPO2 clip

            obj_actor = obj_surrogate + obj_entropy.mean() * self.lambda_entropy# J + KL
            self.optimizer_update(self.act_optimizer, -obj_actor)# 用PG的方式更新actor

            obj_critics += obj_critic.item()# 统计loss
            obj_actors += obj_actor.item()# 统计loss

            # 总更新步数增加一个batch size
            self.total_steps += self.batch_size

        a_std_log = self.act.action_std_log.mean()
        return obj_critics / update_times, obj_actors / update_times, a_std_log.item()

    def update_critic(self, buffer) -> float:
        # 计算Value，该Value是用 behavior critic 算出来的
        with torch.no_grad():
            ftStates, imuStates, actions, logprobs, rewards, undones = buffer
            buffer_size = ftStates.shape[0]
            '''get advantages reward_sums'''
            bs = 2 ** 10  # set a smaller 'batch_size' when out of GPU memory.
            values = [self.cri(ftStates[i:i + bs],imuStates[i:i + bs]) for i in range(0, buffer_size, bs)]
            values = torch.cat(values, dim=0).squeeze(1)

            advantages = self.get_advantages(rewards, undones, values)  # 计算GAE优势函数值

            reward_sums = advantages + values  # reward_sums = reward + next_value = Q
            del rewards, undones, values

            # Trick 1: Advantage Normalization ：： 使用GAE计算完一个batch中的advantage后，计算整个batch中所有advantage的mean和std，然后减均值再除以标准差。
            advantages = (advantages - advantages.mean()) / (advantages.std(dim=0) + 1e-5)
        assert logprobs.shape == advantages.shape == reward_sums.shape == (buffer_size,)

        '''update network'''
        obj_critics = 0.0

        update_times = int(buffer_size * self.repeat_times / self.batch_size)# 使用这些经验更新很多次
        assert update_times >= 1
        for udCnt in range(update_times):
            # 随机获取一个batch的经验
            indices = torch.randint(buffer_size, size=(self.batch_size,), requires_grad=False)
            ftState = ftStates[indices]
            imuState = imuStates[indices]
            action = actions[indices]
            logprob = logprobs[indices]
            advantage = advantages[indices]
            reward_sum = reward_sums[indices]

            # 更新Learning Rate
            if self.usedLRD:
                self.lr_decay(self.total_steps)
                print(self.cri_optimizer.param_groups[0]['lr'])

            # 更新 critic
            value = self.cri(ftState,imuState).squeeze(1)  # V值 ，reward_sum 为 Q 值
            obj_critic = self.criterion(value, reward_sum) # 用smoothL1计算 Q - V ，即TD-error
            self.optimizer_update(self.cri_optimizer, obj_critic) # ctitic的更新用TD-error作为loss，自举

            obj_critics += obj_critic.item() # 统计loss
            # print(obj_critic.item())
            self.criticLoss.append(obj_critic.item())

            # 总更新步数增加一个batch size
            self.total_steps += self.batch_size
            b = 10
            if udCnt % b == 0:
                # 保存critic
                save_path = SavePath+train_name+"/checkpoints/"+f'critic_{int(udCnt / b)}.pth'
                torch.save(agent.cri.state_dict(), save_path)
        return obj_critics / update_times

    def get_advantages(self, rewards: Tensor, undones: Tensor, values: Tensor) -> Tensor:
        advantages = torch.empty_like(values)  # advantage value

        masks = undones * self.gamma
        horizon_len = rewards.shape[0]

        next_value = 0

        advantage = 0  # last_gae_lambda
        for t in range(horizon_len - 1, -1, -1):
            delta = rewards[t] + masks[t] * next_value - values[t]
            advantages[t] = advantage = delta + masks[t] * self.lambda_gae_adv * advantage
            next_value = values[t]
        return advantages

    @staticmethod
    def optimizer_update(optimizer, objective: Tensor):
        optimizer.zero_grad()
        objective.backward()
        # # Trick 7 : Gradient clip
        # nn.utils.clip_grad_norm(objective,0.5)
        optimizer.step()

def make_uniform_speed_potential(agent: AgentPPO) -> Callable[[torch.Tensor, torch.Tensor], float]:
    """Wrap a trained critic as a scalar potential function for OP-CBRS."""
    def _phi(ft_state: torch.Tensor, imu_state: torch.Tensor) -> float:
        with torch.no_grad():
            value = agent.cri(ft_state, imu_state)
            return float(value.reshape(-1)[0].detach().cpu().item())
    return _phi


def apply_sim2sim_transfer(
    agent: AgentPPO,
    test_env: Optional[TestEnv] = None,
    actor_path: Optional[str] = None,
    critic_path: Optional[str] = None,
    source_domain_stats: Optional[Dict[str, float]] = None,
    target_domain_stats: Optional[Dict[str, float]] = None,
) -> AgentPPO:
    """Minimal Sim2Sim transfer utility aligned with the paper.

    It preserves source policy parameters and recalibrates the fuzzy semantics
    used by the target environment.
    """
    if actor_path is not None:
        agent.load(actor_path, critic_path)
    if test_env is not None and source_domain_stats and target_domain_stats:
        test_env.fuzzy.rescale_for_transfer(source_domain_stats, target_domain_stats)
    return agent


if __name__ == "__main__":
    print("uav_inspection_drl.py loaded successfully.")
    print("Core components: Flightmare/ROS env, route planner, PPO-based OSD policy, fuzzy state augmentation, OP-CBRS hooks, Sim2Sim transfer utility.")
